**PM-KUSUM Solar Battery Dispatch — Bilevel Model (10 Households)**

This notebook extends the 1-household model to the full feeder with **10 heterogeneous households** specified in plan §4.2. Each household has a different daily demand (2.1 to 6.2 kWh) reflecting residential, agricultural, and commercial profiles.

- **Lower level (SP1):** all 10 households solved jointly in one LP. Since households have no coupling constraints, the joint LP is equivalent to 10 independent LPs (Calvete & Galé 2007, *independent followers*), but we solve them together for convenience.
- **Upper level (SP2):** DISCOM chooses `lambda_inc` by parametric enumeration; the budget now applies to the feeder-level incentive payment across all 10 households.

Decision variable count: **10 HH × 24 h × 4 vars = 960**. Far beyond Excel Solver's 200-variable limit, which is the main reason for using Python.

In [ ]:
%pip install gurobipy

In [ ]:
from gurobipy import Model, GRB, quicksum
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### 1. Parameters (from Parameters sheet)

Shared across all households — every HH has the same battery spec, PV size, and tariffs.

In [ ]:
# Battery & PV specs (MNRE PM-KUSUM standard, IRENA 2023)
P_pv        = 2      # rooftop solar capacity (kW)
E_bat       = 4      # battery capacity (kWh)
SOC_min     = 0.15   # minimum state-of-charge fraction
P_ch        = 2      # max charge rate (kWh per hour)
P_dis       = 2      # max discharge rate (kWh per hour)
SOC_init    = 2      # battery charge at hour 0 (kWh)
G_max       = 3      # max grid import/export rate (kW)

# Tariffs (RUVN 2023 ToU schedule, JVVNL 2025 feed-in)
lambda_feed = 3.26   # feed-in tariff for surplus PV exported (Rs/kWh)

# DISCOM coordination parameters (plan §3.3).
# Budget is scaled up from 30 (1-HH case) to 300 for the 10-HH feeder.
B_incentive = 300    # max daily incentive budget across all HH (Rs/day)
peak_start  = 18     # peak window start hour (0-indexed: t=18..21)
peak_end    = 21     # peak window end hour (inclusive)

### 2. Households (from plan §4.2)

Ten heterogeneous households representing the rural feeder mix in Rajasthan.

In [ ]:
# Each household: (id, type, daily demand in kWh).
# Calibrated to NITI Aayog 2021 rural consumption ranges.
households = [
    ('HH01', 'Residential',              3.2),
    ('HH02', 'Residential',              2.8),
    ('HH03', 'Residential (vulnerable)', 2.1),
    ('HH04', 'Residential',              3.5),
    ('HH05', 'Residential',              2.6),
    ('HH06', 'Small farm (irrigation)',  5.8),
    ('HH07', 'Small farm (irrigation)',  6.2),
    ('HH08', 'Small shop',               4.1),
    ('HH09', 'Residential',              2.9),
    ('HH10', 'Residential',              3.0),
]
N = len(households)      # number of households
HH_ids = [hh[0] for hh in households]
HH_daily_demand = np.array([hh[2] for hh in households])

print(f'Total feeder demand: {HH_daily_demand.sum():.1f} kWh/day across {N} households')

### 3. Profiles (from Profiles sheet)

Hourly profiles are shared across households (same location, same grid feeder). Only the demand is scaled per household.

In [ ]:
T = 24                    # planning horizon: 24 hours
hours = range(T)          # t = 0..23  (corresponds to Excel t = 1..24)

# PV output per hour in kWh (NREL PVWatts, Jaipur, scaled by P_pv)
pv_output = np.array([0, 0, 0, 0, 0, 0.024, 0.17, 0.44, 0.78, 1.07, 1.31, 1.476,
                      1.53, 1.476, 1.31, 1.07, 0.78, 0.44, 0.12, 0, 0, 0, 0, 0])

# Grid availability a_t (1 = grid up, 0 = load-shedding). PRAYAS ESMI Rajasthan 2023
a_t = np.array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1])

# Time-of-use grid tariff in Rs/kWh (RUVN 2023)
tariff = np.array([4.5]*12 + [5.5, 5.5] + [4.5]*4 + [7.0]*4 + [4.5]*2)

# Household demand shape (normalized daily shape from NITI Aayog 2021)
demand_shape = np.array([0.05, 0.04, 0.04, 0.04, 0.05, 0.08, 0.12, 0.16, 0.20, 0.18, 0.16, 0.15,
                         0.20, 0.18, 0.14, 0.12, 0.13, 0.16, 0.35, 0.38, 0.32, 0.25, 0.18, 0.12])

# Build demand matrix: demand[i, t] = scaled demand for household i at hour t.
# Each HH's demand = shape * (daily_i / sum(shape)), so Sum_t demand[i,t] = daily_i.
demand = np.array([demand_shape * (HH_daily_demand[i] / demand_shape.sum()) for i in range(N)])

# Set of peak hours for incentive payment
peak_hours = list(range(peak_start, peak_end + 1))    # t = 18..21

### 4. Lower-level model: joint 10-household dispatch LP

For a given `lambda_inc`, solve all 10 households simultaneously:
$$\min \;\; \sum_i \left[ \sum_t \lambda_t \cdot g_{i,t} \;-\; \lambda_{inc} \sum_{t \in \text{Peak}} dis_{i,t} \;-\; \lambda_{feed} \sum_t feed_{i,t} \right]$$

Decision variables are indexed by `(i, t)`: household `i = 0..9`, hour `t = 0..23`. Total: 10 × 24 × 4 = 960 decision variables.

In [ ]:
def solve_all_HH_LP(lambda_inc, verbose=False):
    """Solve the joint 10-HH dispatch LP for a given incentive rate.
    Returns: (total objective, dict of per-HH optimal decision arrays)."""

    m = Model('HH_Dispatch_10HH')
    if not verbose:
        m.setParam('OutputFlag', 0)

    # decision variables indexed by (household, hour)
    ch   = m.addVars(N, T, name='ch')
    dis  = m.addVars(N, T, name='dis')
    g    = m.addVars(N, T, name='g')
    feed = m.addVars(N, T, name='feed')
    s    = m.addVars(N, T, name='s')

    # objective: total grid cost - incentive revenue - feed-in revenue across all HH
    m.setObjective(
        quicksum(tariff[t] * g[i,t] for i in range(N) for t in hours)
        - lambda_inc  * quicksum(dis[i,t]  for i in range(N) for t in peak_hours)
        - lambda_feed * quicksum(feed[i,t] for i in range(N) for t in hours),
        GRB.MINIMIZE
    )

    # per-HH energy balance: PV + dis + g = demand + ch + feed
    m.addConstrs((
        pv_output[t] + dis[i,t] + g[i,t] == demand[i,t] + ch[i,t] + feed[i,t]
        for i in range(N) for t in hours
    ), name='balance')

    # per-HH SOC dynamics
    m.addConstrs((s[i,0] == SOC_init + ch[i,0] - dis[i,0] for i in range(N)), name='soc_init')
    m.addConstrs((
        s[i,t] == s[i,t-1] + ch[i,t] - dis[i,t]
        for i in range(N) for t in range(1, T)
    ), name='soc_dynamics')

    # per-HH SOC bounds
    m.addConstrs((s[i,t] >= SOC_min * E_bat for i in range(N) for t in hours), name='soc_lower')
    m.addConstrs((s[i,t] <= E_bat           for i in range(N) for t in hours), name='soc_upper')

    # per-HH charge / discharge rate limits
    m.addConstrs((ch[i,t]  <= P_ch  for i in range(N) for t in hours), name='ch_limit')
    m.addConstrs((dis[i,t] <= P_dis for i in range(N) for t in hours), name='dis_limit')

    # per-HH grid import and feed-in only when grid is available
    m.addConstrs((g[i,t]    <= G_max * a_t[t] for i in range(N) for t in hours), name='g_avail')
    m.addConstrs((feed[i,t] <= G_max * a_t[t] for i in range(N) for t in hours), name='feed_avail')

    m.optimize()

    # extract optimal values into (N × T) numpy arrays
    solution = {
        'ch':   np.array([[ch[i,t].X   for t in hours] for i in range(N)]),
        'dis':  np.array([[dis[i,t].X  for t in hours] for i in range(N)]),
        'g':    np.array([[g[i,t].X    for t in hours] for i in range(N)]),
        'feed': np.array([[feed[i,t].X for t in hours] for i in range(N)]),
        's':    np.array([[s[i,t].X    for t in hours] for i in range(N)]),
    }
    return m.ObjVal, solution

### 5. Solve the lower level for one specific lambda_inc

Test at `lambda_inc = 1.5` to match the Excel v4 test case.

In [ ]:
lambda_inc_test = 1.5
obj, sol = solve_all_HH_LP(lambda_inc_test)

print(f'10-HH total net daily cost: {obj:.2f} Rs')
print(f'Total peak discharge (t=19-22): {sol["dis"][:, peak_hours].sum():.2f} kWh')
print(f'Total feed-in energy:           {sol["feed"].sum():.2f} kWh')
print(f'Total grid import:              {sol["g"].sum():.2f} kWh')

In [ ]:
# per-household summary at this lambda_inc
per_hh_summary = []
for i in range(N):
    grid_cost = (tariff * sol['g'][i]).sum()
    inc_rev   = lambda_inc_test * sol['dis'][i, peak_hours].sum()
    feed_rev  = lambda_feed * sol['feed'][i].sum()
    per_hh_summary.append({
        'HH_ID':        HH_ids[i],
        'Type':         households[i][1],
        'Daily_Demand': HH_daily_demand[i],
        'Grid_Cost':    round(grid_cost, 2),
        'Inc_Rev':      round(inc_rev, 2),
        'Feed_Rev':     round(feed_rev, 2),
        'Net_Cost':     round(grid_cost - inc_rev - feed_rev, 2),
        'Peak_Disch':   round(sol['dis'][i, peak_hours].sum(), 2),
        'Feed_kWh':     round(sol['feed'][i].sum(), 2),
    })

pd.DataFrame(per_hh_summary).set_index('HH_ID')

### 5b. Precise optimal lambda_inc (beyond the discrete grid)

The discrete grid in Step 6 may miss the true optimum, especially near the budget-binding boundary. Because the household response is a step function (zero peak discharge at λ_inc=0, saturating at the physical maximum for any λ_inc > 0), the **precise optimal λ\*** is the largest rate that still satisfies the DISCOM budget constraint:

$$\lambda^* \;=\; \frac{B_{\mathrm{incentive}}}{\sum_i \sum_{t \in \text{Peak}} dis_{i,t}^*(\lambda)}$$

where the denominator is the saturated peak discharge (any λ_inc > 0 gives the same value). We compute λ\* in two steps:

1. **Probe the saturation level** by solving the LP at a small positive λ (e.g., 0.01) — this gives total peak discharge at saturation.
2. **Apply the closed-form formula** above. Then verify by solving the LP one more time at λ\* and checking the budget binds exactly.

In [ ]:
# Step 1: probe saturation peak discharge at any small positive lambda_inc
_, sol_probe = solve_all_HH_LP(0.01)
saturated_peak_discharge = sol_probe['dis'][:, peak_hours].sum()
print(f'Saturated peak discharge (10 HH at any lambda > 0): {saturated_peak_discharge:.4f} kWh')

# Step 2: closed-form precise optimum
lambda_star = B_incentive / saturated_peak_discharge
print(f'\nPrecise optimal lambda* = B / peak_discharge = {B_incentive} / {saturated_peak_discharge:.4f}')
print(f'                       = {lambda_star:.6f} Rs/kWh')

# Step 3: verify by solving the LP at lambda* and confirming budget binds exactly
obj_star, sol_star = solve_all_HH_LP(lambda_star)
verified_peak = sol_star['dis'][:, peak_hours].sum()
verified_budget_used = lambda_star * verified_peak

print(f'\nVerification at lambda* = {lambda_star:.4f}:')
print(f'  Total HH net cost:      {obj_star:.2f} Rs')
print(f'  Total peak discharge:   {verified_peak:.4f} kWh')
print(f'  Budget used:            {verified_budget_used:.4f} Rs (cap = {B_incentive})')
print(f'  Budget binds exactly:   {abs(verified_budget_used - B_incentive) < 1e-6}')

# Compare to the best discrete-grid value
print(f'\nFor reference, the best DISCRETE value from Step 6 was lambda = 3 Rs/kWh.')
print(f'The precise optimum is {lambda_star:.4f} Rs/kWh — that is {(lambda_star/3 - 1)*100:.1f}% higher,')
print(f'extracting an additional {(lambda_star - 3) * verified_peak:.2f} Rs/day of incentive payment for the same peak reduction.')

### 6. Upper-level model: DISCOM coordination via parametric enumeration

Sweep `lambda_inc` across candidate values and record aggregate response. DISCOM's optimal rate maximizes feeder peak discharge while staying within the 300 Rs/day budget.

In [ ]:
lambda_grid = [0, 0.01, 0.2, 0.5, 1, 2, 3, 4, 5, 6, 7, 8]

rows = []
for lam in lambda_grid:
    obj, sol = solve_all_HH_LP(lam)
    peak_dis    = sol['dis'][:, peak_hours].sum()
    budget_used = lam * peak_dis
    rows.append({
        'lambda_inc':     lam,
        'Total_HH_cost':  round(obj, 2),
        'Peak_Disch':     round(peak_dis, 2),
        'Feed_in':        round(sol['feed'].sum(), 2),
        'Budget_Used':    round(budget_used, 2),
        'Within_Budget':  budget_used <= B_incentive,
    })

sweep = pd.DataFrame(rows)
sweep

In [ ]:
# DISCOM's optimal choice: highest lambda_inc that stays within budget
feasible = sweep[sweep['Within_Budget']]
optimal  = feasible.loc[feasible['Total_HH_cost'].idxmin()]

print(f'Optimal lambda_inc (within budget, min total HH cost): {optimal["lambda_inc"]} Rs/kWh')
print(f'  Feeder peak discharge: {optimal["Peak_Disch"]} kWh')
print(f'  Budget used:           {optimal["Budget_Used"]} Rs (limit = {B_incentive} Rs)')
print(f'  Total HH net cost:     {optimal["Total_HH_cost"]} Rs')

### 7. Visualization: Pareto frontier and per-HH heterogeneity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# left: total household cost vs incentive rate
axes[0].plot(sweep['lambda_inc'], sweep['Total_HH_cost'], marker='o')
axes[0].set_xlabel('Incentive rate lambda_inc (Rs/kWh)')
axes[0].set_ylabel('Total 10-HH daily net cost (Rs)')
axes[0].set_title('Feeder cost vs. incentive rate')
axes[0].axhline(0, color='gray', linewidth=0.5)

# right: budget used vs incentive rate, with budget line
axes[1].plot(sweep['lambda_inc'], sweep['Budget_Used'], marker='o', color='C1')
axes[1].axhline(B_incentive, color='red', linestyle='--', label=f'Budget = {B_incentive} Rs')
axes[1].set_xlabel('Incentive rate lambda_inc (Rs/kWh)')
axes[1].set_ylabel('DISCOM budget used (Rs)')
axes[1].set_title('Budget usage vs. incentive rate')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# heterogeneity visualization: how each HH's net cost varies with lambda_inc
cost_matrix = np.zeros((N, len(lambda_grid)))
for j, lam in enumerate(lambda_grid):
    _, sol = solve_all_HH_LP(lam)
    for i in range(N):
        gc  = (tariff * sol['g'][i]).sum()
        ir  = lam * sol['dis'][i, peak_hours].sum()
        fr  = lambda_feed * sol['feed'][i].sum()
        cost_matrix[i, j] = gc - ir - fr

plt.figure(figsize=(10, 5))
for i in range(N):
    plt.plot(lambda_grid, cost_matrix[i], marker='o', label=f'{HH_ids[i]} ({HH_daily_demand[i]} kWh)')
plt.xlabel('Incentive rate lambda_inc (Rs/kWh)')
plt.ylabel('HH daily net cost (Rs)')
plt.title('Per-household cost response to incentive rate')
plt.axhline(0, color='gray', linewidth=0.5)
plt.legend(loc='lower left', fontsize=8, ncol=2)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()